In [ ]:
!pip -q install ultralytics scikit-learn pyyaml pandas

In [ ]:
#import libraries
import shutil
import random
from pathlib import Path
import yaml
import pandas as pd
from sklearn.model_selection import KFold
from ultralytics import YOLO

In [ ]:
# Select Platform
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Project Path
if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/Final_Thesis")

else:
    PROJECT_ROOT = Path.cwd()
print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)


Running in Colab: True
Project root: /content/drive/MyDrive/Final_Thesis


In [ ]:
# Dataset
DATASET_ROOT = PROJECT_ROOT / "dataset"
RUNS_PATH = PROJECT_ROOT / "runs"
CV_ROOT = PROJECT_ROOT / "CrossVal"

TRAIN_IMAGES_DIR = DATASET_ROOT / "train" / "images"
TRAIN_LABELS_DIR = DATASET_ROOT / "train" / "labels"

VAL_IMAGES_DIR = DATASET_ROOT / "valid" / "images"
VAL_LABELS_DIR = DATASET_ROOT / "valid" / "labels"

TEST_IMAGES_DIR = DATASET_ROOT / "test" / "images"
TEST_LABELS_DIR = DATASET_ROOT / "test" / "labels"

In [ ]:
#Model Configuration
MODEL_WEIGHTS = "yolo11s.pt"

N_FOLDS = 5
EPOCHS = 100
BATCH = 8
IMGSZ = 640
SEED = 42

NC = 1
CLASS_NAMES = ["bus"]

IMG_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

random.seed(SEED)

RUNS_PATH.mkdir(parents=True, exist_ok=True)
CV_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Helper Function for images and labels
def get_image_files(images_dir):
    """
    Find all image files inside a given image directory.

    Parameters
    ----------
    images_dir : str or Path
        Path to the image directory.

    Returns
    -------
    list
        Sorted list of image file paths.
    """

    images_dir = Path(images_dir)

    if not images_dir.exists():
        return []

    files = []

    for ext in IMG_EXTS:
        files.extend(images_dir.rglob(f"*{ext}"))
        files.extend(images_dir.rglob(f"*{ext.upper()}"))

    return sorted(files)


def get_label_path(img_path):
    """
    Get the corresponding YOLO label path for an image.

    """

    img_path = Path(img_path)

    split_name = img_path.parent.parent.name

    label_path = (
        DATASET_ROOT
        / split_name
        / "labels"
        / f"{img_path.stem}.txt"
    )

    return label_path


def safe_copy(src, dst):
    """
    Copy a file safely and create destination folder if needed.
    """

    src = Path(src)
    dst = Path(dst)

    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


def copy_image_and_label(img_path, dst_img_dir, dst_lbl_dir):
    """
    Copy one image and its corresponding label file.

    If the label file is missing, an empty label file is created.
    """

    img_path = Path(img_path)

    label_path = get_label_path(img_path)

    dst_img = Path(dst_img_dir) / img_path.name
    dst_lbl = Path(dst_lbl_dir) / f"{img_path.stem}.txt"

    safe_copy(img_path, dst_img)

    dst_lbl.parent.mkdir(parents=True, exist_ok=True)

    if label_path.exists():
        safe_copy(label_path, dst_lbl)
    else:
        dst_lbl.touch()


def create_data_yaml(fold_dir):
    """
    Create a YOLO data.yaml file for a specific fold.
    """

    fold_yaml = Path(fold_dir) / "data.yaml"

    fold_yaml_content = {
        "path": str(fold_dir),
        "train": "train/images",
        "val": "valid/images",
        "nc": NC,
        "names": CLASS_NAMES
    }

    with open(fold_yaml, "w") as f:
        yaml.safe_dump(fold_yaml_content, f, sort_keys=False)

    return fold_yaml


def print_dataset_summary():
    """
    Print dataset folder status and number of images.
    """

    print("Checking dataset folders...\n")

    print("Dataset root:", DATASET_ROOT)
    print("Train images folder exists:", TRAIN_IMAGES_DIR.exists())
    print("Valid images folder exists:", VAL_IMAGES_DIR.exists())
    print("Test images folder exists:", TEST_IMAGES_DIR.exists())

    train_images = get_image_files(TRAIN_IMAGES_DIR)
    val_images = get_image_files(VAL_IMAGES_DIR)
    test_images = get_image_files(TEST_IMAGES_DIR)

    print("\nImages found:")
    print("Train images:", len(train_images))
    print("Validation images:", len(val_images))
    print("Test images:", len(test_images))

    if len(train_images) > 0:
        print("\nSample train image:", train_images[0])

    if len(val_images) > 0:
        print("Sample validation image:", val_images[0])

    if len(test_images) > 0:
        print("Sample test image:", test_images[0])

    return train_images, val_images, test_images


In [ ]:
# Check Dataset
train_images, val_images, test_images = print_dataset_summary()

all_images = train_images + val_images

print("\nTotal images used for cross-validation:", len(all_images))


if len(all_images) == 0:
    raise ValueError(
        "No images found. Please check your dataset folder structure. "
    )


if len(all_images) < N_FOLDS:
    raise ValueError(
        f"You have only {len(all_images)} images, "
        f"but N_FOLDS is set to {N_FOLDS}. "
        "Please reduce N_FOLDS."
    )


Checking dataset folders...

Dataset root: /content/drive/MyDrive/Final_Thesis/dataset
Train images folder exists: True
Valid images folder exists: True
Test images folder exists: True

Images found:
Train images: 161
Validation images: 8
Test images: 4

Sample train image: /content/drive/MyDrive/Final_Thesis/dataset/train/images/Bålsta_1_mp4-0000_jpg.rf.EALAKPgKsvOFET0A5UUf.jpg
Sample validation image: /content/drive/MyDrive/Final_Thesis/dataset/valid/images/Balsta_1_trim_mp4-0001_jpg.rf.84133b9d03529ab5fd9c45bacf33a1ec.jpg
Sample test image: /content/drive/MyDrive/Final_Thesis/dataset/test/images/Balsta_1_trim_mp4-0013_jpg.rf.7aa80a12b8e66a66cfa9ef08a316d103.jpg

Total images used for cross-validation: 169


In [ ]:
# K-Fold Cross-Validation
kf = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

fold_summaries = []


for fold_idx, (train_idx, val_idx) in enumerate(
    kf.split(all_images),
    start=1
):
    print("\n" + "=" * 70)
    print(f"Starting fold {fold_idx}/{N_FOLDS}")
    print("=" * 70)

    fold_dir = CV_ROOT / f"fold_{fold_idx}"

    # Remove previous fold folder if it exists
    if fold_dir.exists():
        shutil.rmtree(fold_dir)

    # Create fold-specific directories
    fold_train_img_dir = fold_dir / "train" / "images"
    fold_train_lbl_dir = fold_dir / "train" / "labels"

    fold_val_img_dir = fold_dir / "valid" / "images"
    fold_val_lbl_dir = fold_dir / "valid" / "labels"

    fold_train_img_dir.mkdir(parents=True, exist_ok=True)
    fold_train_lbl_dir.mkdir(parents=True, exist_ok=True)

    fold_val_img_dir.mkdir(parents=True, exist_ok=True)
    fold_val_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Select images for this fold
    fold_train_imgs = [all_images[i] for i in train_idx]
    fold_val_imgs = [all_images[i] for i in val_idx]

    print(f"Fold {fold_idx} training images: {len(fold_train_imgs)}")
    print(f"Fold {fold_idx} validation images: {len(fold_val_imgs)}")

    # Copy training images and labels
    for img_path in fold_train_imgs:
        copy_image_and_label(
            img_path,
            fold_train_img_dir,
            fold_train_lbl_dir
        )

    # Copy validation images and labels
    for img_path in fold_val_imgs:
        copy_image_and_label(
            img_path,
            fold_val_img_dir,
            fold_val_lbl_dir
        )

    # Create data.yaml for this fold
    fold_yaml = create_data_yaml(fold_dir)

    print("Fold YAML created:", fold_yaml)

    # Train a fresh YOLO model for this fold
    model = YOLO(MODEL_WEIGHTS)

    model.train(
        data=str(fold_yaml),
        epochs=EPOCHS,
        batch=BATCH,
        imgsz=IMGSZ,
        project=str(RUNS_PATH),
        name=f"yolo11s_bus_fold_{fold_idx}",
        seed=SEED,
        verbose=True
    )

    # Load best model weights from this fold
    best_weights = (
        RUNS_PATH
        / f"yolo11s_bus_fold_{fold_idx}"
        / "weights"
        / "best.pt"
    )

    if not best_weights.exists():
        raise FileNotFoundError(
            f"Best weights not found for fold {fold_idx}: {best_weights}"
        )

    best_model = YOLO(str(best_weights))

    # Validate best model on this fold's validation set
    metrics = best_model.val(
        data=str(fold_yaml),
        split="val"
    )

    fold_result = {
        "fold": fold_idx,
        "train_images": len(fold_train_imgs),
        "val_images": len(fold_val_imgs),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
    }

    fold_summaries.append(fold_result)

    print(f"\nFold {fold_idx} results:")
    print(fold_result)


Starting fold 1/5
Fold 1 training images: 135
Fold 1 validation images: 34
Fold YAML created: /content/drive/MyDrive/Final_Thesis/CrossVal/fold_1/data.yaml
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Final_Thesis/CrossVal/fold_1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixu

In [ ]:
# Save Cross-Validation Results
results_df = pd.DataFrame(fold_summaries)

mean_row = {
    "fold": "mean",
    "train_images": results_df["train_images"].mean(),
    "val_images": results_df["val_images"].mean(),
    "precision": results_df["precision"].mean(),
    "recall": results_df["recall"].mean(),
    "map50": results_df["map50"].mean(),
    "map50_95": results_df["map50_95"].mean(),
}

std_row = {
    "fold": "std",
    "train_images": results_df["train_images"].std(),
    "val_images": results_df["val_images"].std(),
    "precision": results_df["precision"].std(),
    "recall": results_df["recall"].std(),
    "map50": results_df["map50"].std(),
    "map50_95": results_df["map50_95"].std(),
}

results_df = pd.concat(
    [
        results_df,
        pd.DataFrame([mean_row, std_row])
    ],
    ignore_index=True
)

csv_path = CV_ROOT / "cross_validation_results.csv"

results_df.to_csv(csv_path, index=False)

print("\nCross-validation finished successfully.")
print("\nResults:")
print(results_df)
print(f"\nResults saved to: {csv_path}")